# 06 RQ5 Deployment-Oriented Trade-off Analysis

This notebook evaluates the deployment-oriented trade-offs of the trained CNN backbone models.

It performs:
- checkpoint loading
- predictive performance evaluation on PlantVillage and PlantDoc
- parameter count and model size measurement
- inference latency benchmarking
- deployment suitability analysis
- export of tables and figures

Outputs:
- Table 9: deployment efficiency comparison
- Table 10: deployment suitability comparison
- Figure 9: accuracy-efficiency Pareto analysis
- Figure 10: end-to-end response breakdown
- ZIP archive of RQ5 outputs

In [1]:
# ----------------------------------------
# Section 1: Imports
# ----------------------------------------

import os
import json
import time
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [2]:
# ----------------------------------------
# Section 2: Reproducibility setup
# ----------------------------------------

SEED = 42

def seed_everything(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility across Python, NumPy, and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    """
    Ensure each DataLoader worker uses a deterministic seed.
    """
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(SEED)

print("Reproducibility setup completed")
print(f"Global seed: {SEED}")

Reproducibility setup completed
Global seed: 42


In [3]:
# ----------------------------------------
# Section 3: Configuration
# ----------------------------------------

CONFIG = {
    "seed": SEED,
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 0,
    "plantvillage_root": "/kaggle/input/datasets/thedataeng/plantvillage",
    "plantdoc_root": "/kaggle/input/datasets/thedataeng/plantdoc",
    
    # Update this if your Kaggle dataset input name differs
    "checkpoint_root": "/kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs",
    
    "output_root": "/kaggle/working/thesis_outputs/rq5_deployment",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_ROOT = Path(CONFIG["output_root"])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAMES = ["resnet50", "efficientnet_b0", "mobilenet_v2"]

print("Configuration loaded")
print(f"Device: {DEVICE}")
print(f"Checkpoint root: {CONFIG['checkpoint_root']}")
print(f"Output root: {OUTPUT_ROOT}")

Configuration loaded
Device: cuda
Checkpoint root: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs
Output root: /kaggle/working/thesis_outputs/rq5_deployment


In [4]:
# ----------------------------------------
# Section 4: Helper functions
# ----------------------------------------

def ensure_dir(path: Path) -> Path:
    """
    Create a directory if it does not exist and return the Path object.
    """
    path.mkdir(parents=True, exist_ok=True)
    return path

def get_eval_transform(image_size: int = 224):
    """
    Create the evaluation transform used across RQ5 experiments.
    """
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

def create_model(model_name: str, num_classes: int) -> nn.Module:
    """
    Create the selected backbone model and replace the classification head.
    """
    if model_name == "resnet50":
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model name: {model_name}")

    return model.to(DEVICE)

def checkpoint_path(model_name: str) -> Path:
    """
    Return the checkpoint path for a given model name.
    """
    return Path(CONFIG["checkpoint_root"]) / "checkpoints" / f"{model_name}_seed{SEED}_best.pt"

def count_parameters(model: nn.Module) -> int:
    """
    Count the number of trainable parameters in the model.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def model_size_mb(model: nn.Module, temp_name: str) -> float:
    """
    Save model weights temporarily to estimate file size in MB.
    """
    temp_path = OUTPUT_ROOT / temp_name
    torch.save(model.state_dict(), temp_path)
    size_mb = temp_path.stat().st_size / (1024 * 1024)
    temp_path.unlink(missing_ok=True)
    return float(size_mb)

@torch.no_grad()
def predict_loader(model: nn.Module, loader: DataLoader):
    """
    Run model inference on a DataLoader and return probabilities, predictions, and labels.
    """
    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE)

        logits = model(images)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(labels.numpy())

    return (
        np.vstack(all_probs),
        np.concatenate(all_preds),
        np.concatenate(all_labels),
    )

def evaluate_model(model: nn.Module, loader: DataLoader) -> dict:
    """
    Evaluate the model and return standard classification metrics.
    """
    probs, preds, labels = predict_loader(model, loader)

    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "preds": preds,
        "labels": labels,
        "probs": probs,
    }

def pretty_metric(x: float) -> float:
    """
    Round a metric value for cleaner table presentation.
    """
    return round(float(x), 4)

def save_table(df: pd.DataFrame, name: str) -> None:
    """
    Save a DataFrame as CSV in the tables directory.
    """
    table_dir = ensure_dir(OUTPUT_ROOT / "tables")
    csv_path = table_dir / f"{name}.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved table: {csv_path}")

def save_figure(fig: plt.Figure, name: str) -> None:
    """
    Save a matplotlib figure as PDF in the figures directory.
    """
    fig_dir = ensure_dir(OUTPUT_ROOT / "figures")
    pdf_path = fig_dir / f"{name}.pdf"
    fig.tight_layout()
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {pdf_path}")

def deployment_suitability(latency_ms: float, size_mb: float) -> str:
    """
    Assign a practical deployment suitability category based on model size and latency.
    These categories are engineering-oriented interpretations, not universal thresholds.
    """
    if latency_ms < 60 and size_mb < 20:
        return "High"
    elif latency_ms < 100 and size_mb < 50:
        return "Medium"
    return "Low"

print('Done')

Done


In [5]:
# ----------------------------------------
# Section 5: Dataset loading
# ----------------------------------------

pv_root = Path(CONFIG["plantvillage_root"])
pd_root = Path(CONFIG["plantdoc_root"])

test_dir = pv_root / "test"

assert test_dir.exists(), f"Missing PlantVillage test directory: {test_dir}"
assert pd_root.exists(), f"Missing PlantDoc directory: {pd_root}"

eval_tfms = get_eval_transform(CONFIG["image_size"])

test_dataset = datasets.ImageFolder(test_dir, transform=eval_tfms)
plantdoc_dataset = datasets.ImageFolder(pd_root, transform=eval_tfms)

generator = torch.Generator()
generator.manual_seed(SEED)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

plantdoc_loader = DataLoader(
    plantdoc_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

CLASS_NAMES = test_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print("Datasets loaded successfully")
print(f"PlantVillage test samples: {len(test_dataset)}")
print(f"PlantDoc samples:          {len(plantdoc_dataset)}")
print(f"Number of classes:         {NUM_CLASSES}")

Datasets loaded successfully
PlantVillage test samples: 5553
PlantDoc samples:          2555
Number of classes:         27


In [6]:
# ----------------------------------------
# Section 6: Checkpoint loading
# ----------------------------------------

models_loaded = {}

for idx, model_name in enumerate(MODEL_NAMES, start=1):
    print(f"Loading checkpoint {idx}/{len(MODEL_NAMES)}: {model_name}")

    ckpt_path = checkpoint_path(model_name)
    assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"

    model = create_model(model_name, NUM_CLASSES)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    models_loaded[model_name] = model

    print(f"Loaded checkpoint: {ckpt_path}")

print("All checkpoints loaded successfully")

Loading checkpoint 1/3: resnet50
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/resnet50_seed42_best.pt
Loading checkpoint 2/3: efficientnet_b0
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/efficientnet_b0_seed42_best.pt
Loading checkpoint 3/3: mobilenet_v2
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/mobilenet_v2_seed42_best.pt
All checkpoints loaded successfully


In [8]:
# ----------------------------------------
# Section 7: Deployment benchmarking
# ----------------------------------------

deployment_rows = []

def expert_decision(conf: float, ent: float) -> tuple:
    """
    Uncertainty-aware expert decision-support rule.
    """
    if conf >= 0.75 and ent <= 0.30:
        return "Accept", "Recommend treatment or monitoring"

    if conf >= 0.60 and ent <= 0.40:
        return "Monitor", "Monitor and retake if symptoms persist"

    if conf >= 0.50 and ent <= 0.50:
        return "Retake", "Request a clearer image"

    return "Review", "Seek expert review"


for idx, model_name in enumerate(MODEL_NAMES, start=1):
    print(f"Benchmarking model {idx}/{len(MODEL_NAMES)}: {model_name}")

    model = models_loaded[model_name]

    print("  Evaluating on PlantVillage test set")
    pv_metrics = evaluate_model(model, test_loader)

    print("  Evaluating on PlantDoc")
    pd_metrics = evaluate_model(model, plantdoc_loader)

    print("  Measuring parameter count and model size")
    params_m = count_parameters(model) / 1e6
    size_mb = model_size_mb(model, f"temp_{model_name}_weights.pt")

    print("  Measuring deployment pipeline latency with batch size 1")

    model.eval()
    warmups = 10
    runs = 50

    sample_images, _ = next(iter(test_loader))
    sample_image = sample_images[0].unsqueeze(0).to(DEVICE)

    preprocess_times = []
    inference_times = []
    decision_times = []

    with torch.no_grad():
        for _ in range(warmups):
            _ = model(sample_image)

    print("  Warm-up completed")

    for run_idx in range(runs):

        # 1. Preprocessing
        t0 = time.perf_counter()

        x = sample_image.clone().to(DEVICE)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        t1 = time.perf_counter()
        preprocess_times.append((t1 - t0) * 1000.0)

        # 2. Inference
        with torch.no_grad():
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            t2 = time.perf_counter()
            outputs = model(x)

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            t3 = time.perf_counter()

        inference_times.append((t3 - t2) * 1000.0)

        # 3. Decision logic
        t4 = time.perf_counter()

        probs = torch.softmax(outputs, dim=1)
        conf = torch.max(probs, dim=1).values.item()

        entropy = -torch.sum(
            probs * torch.log(probs + 1e-8),
            dim=1
        ).item()

        ent = entropy / np.log(probs.shape[1])

        decision, recommendation = expert_decision(conf, ent)

        t5 = time.perf_counter()
        decision_times.append((t5 - t4) * 1000.0)

        if (run_idx + 1) % 10 == 0:
            print(f"  Completed {run_idx + 1}/{runs} latency runs")

    avg_preprocess_ms = np.mean(preprocess_times)
    avg_inference_ms = np.mean(inference_times)
    avg_decision_ms = np.mean(decision_times)
    avg_pipeline_ms = avg_preprocess_ms + avg_inference_ms + avg_decision_ms

    print("  Latency summary:")
    print(f"    Preprocessing: {avg_preprocess_ms:.3f} ms")
    print(f"    Inference:     {avg_inference_ms:.3f} ms")
    print(f"    Decision:      {avg_decision_ms:.3f} ms")
    print(f"    Total:         {avg_pipeline_ms:.3f} ms")

    deployment_rows.append({
        "Model": model_name,
        "Parameters (M)": pretty_metric(params_m),
        "Model Size (MB)": pretty_metric(size_mb),
        "Avg. Preprocess Time (ms)": pretty_metric(avg_preprocess_ms),
        "Avg. Inference Time (ms)": pretty_metric(avg_inference_ms),
        "Avg. Decision Time (ms)": pretty_metric(avg_decision_ms),
        "Avg. Pipeline Time (ms)": pretty_metric(avg_pipeline_ms),
        "PlantVillage Accuracy (%)": pretty_metric(pv_metrics["accuracy"] * 100),
        "PlantVillage F1": pretty_metric(pv_metrics["f1"]),
        "PlantDoc Accuracy (%)": pretty_metric(pd_metrics["accuracy"] * 100),
        "PlantDoc F1": pretty_metric(pd_metrics["f1"]),
    })

print("Deployment benchmarking completed successfully")

Benchmarking model 1/3: resnet50
  Evaluating on PlantVillage test set
  Evaluating on PlantDoc
  Measuring parameter count and model size
  Measuring deployment pipeline latency with batch size 1
  Warm-up completed
  Completed 10/50 latency runs
  Completed 20/50 latency runs
  Completed 30/50 latency runs
  Completed 40/50 latency runs
  Completed 50/50 latency runs
  Latency summary:
    Preprocessing: 0.216 ms
    Inference:     6.489 ms
    Decision:      1.344 ms
    Total:         8.048 ms
Benchmarking model 2/3: efficientnet_b0
  Evaluating on PlantVillage test set
  Evaluating on PlantDoc
  Measuring parameter count and model size
  Measuring deployment pipeline latency with batch size 1
  Warm-up completed
  Completed 10/50 latency runs
  Completed 20/50 latency runs
  Completed 30/50 latency runs
  Completed 40/50 latency runs
  Completed 50/50 latency runs
  Latency summary:
    Preprocessing: 0.075 ms
    Inference:     8.252 ms
    Decision:      0.184 ms
    Total:     

In [9]:
# ----------------------------------------
# Section 8: Save Table 9 - Deployment efficiency comparison
# ----------------------------------------

table9_df = pd.DataFrame(deployment_rows)

pretty_names = {
    "resnet50": "ResNet50",
    "efficientnet_b0": "EfficientNet-B0",
    "mobilenet_v2": "MobileNetV2",
}

table9_df["Model"] = table9_df["Model"].map(pretty_names)

# Arrange columns in a clear publication-ready order
table9_columns = [
    "Model",
    "Parameters (M)",
    "Model Size (MB)",
    "Avg. Preprocess Time (ms)",
    "Avg. Inference Time (ms)",
    "Avg. Decision Time (ms)",
    "Avg. Pipeline Time (ms)",
    "PlantVillage Accuracy (%)",
    "PlantVillage F1",
    "PlantDoc Accuracy (%)",
    "PlantDoc F1",
]

table9_df = table9_df[table9_columns]

save_table(table9_df, "Table_9_Deployment_Efficiency_Comparison")

print("Table 9 saved successfully")
display(table9_df)

Saved table: /kaggle/working/thesis_outputs/rq5_deployment/tables/Table_9_Deployment_Efficiency_Comparison.csv
Table 9 saved successfully


,Model,Parameters (M),Model Size (MB),Avg. Preprocess Time (ms),Avg. Inference Time (ms),Avg. Decision Time (ms),Avg. Pipeline Time (ms),PlantVillage Accuracy (%),PlantVillage F1,PlantDoc Accuracy (%),PlantDoc F1
0,ResNet50,23.5634,90.1947,0.2156,6.4886,1.3441,8.0483,99.4958,0.9921,20.2348,0.1869
1,EfficientNet-B0,4.0421,15.7117,0.0745,8.2520,0.1842,8.5107,99.6218,0.9940,17.6517,0.1457
2,MobileNetV2,2.2585,8.8542,0.0714,5.3118,0.1875,5.5707,99.4778,0.9927,22.2309,0.1712


In [13]:
# ----------------------------------------
# Section 9: Save Table 10 - Measured deployment ranking
# ----------------------------------------

table10_df = table9_df.copy()

# Convert needed columns to float
numeric_cols = [
    "Model Size (MB)",
    "Avg. Preprocess Time (ms)",
    "Avg. Inference Time (ms)",
    "Avg. Decision Time (ms)",
    "Avg. Pipeline Time (ms)",
    "PlantVillage F1",
    "PlantDoc F1",
]

for col in numeric_cols:
    table10_df[col] = table10_df[col].astype(float)

# Rank models using only measured values
# Lower latency and size are better; higher F1 is better
table10_df["Latency Rank"] = table10_df["Avg. Pipeline Time (ms)"].rank(
    ascending=True,
    method="min"
)

table10_df["Size Rank"] = table10_df["Model Size (MB)"].rank(
    ascending=True,
    method="min"
)

table10_df["PlantVillage F1 Rank"] = table10_df["PlantVillage F1"].rank(
    ascending=False,
    method="min"
)

table10_df["PlantDoc F1 Rank"] = table10_df["PlantDoc F1"].rank(
    ascending=False,
    method="min"
)

# Overall measured trade-off rank
table10_df["Mean Rank"] = table10_df[
    [
        "Latency Rank",
        "Size Rank",
        "PlantVillage F1 Rank",
        "PlantDoc F1 Rank",
    ]
].mean(axis=1)

table10_df = table10_df.sort_values("Mean Rank", ascending=True)

table10_df = table10_df[
    [
        "Model",
        "Avg. Pipeline Time (ms)",
        "Model Size (MB)",
        "PlantVillage F1",
        "PlantDoc F1",
        "Latency Rank",
        "Size Rank",
        "PlantVillage F1 Rank",
        "PlantDoc F1 Rank",
        "Mean Rank",
    ]
]

save_table(table10_df, "Table_10_Measured_Deployment_Ranking")

print("Table 10 saved successfully")
display(table10_df)

Saved table: /kaggle/working/thesis_outputs/rq5_deployment/tables/Table_10_Measured_Deployment_Ranking.csv
Table 10 saved successfully


,Model,Avg. Pipeline Time (ms),Model Size (MB),PlantVillage F1,PlantDoc F1,Latency Rank,Size Rank,PlantVillage F1 Rank,PlantDoc F1 Rank,Mean Rank
2,MobileNetV2,5.5707,8.8542,0.9927,0.1712,1.0,1.0,2.0,2.0,1.50
0,ResNet50,8.0483,90.1947,0.9921,0.1869,2.0,3.0,3.0,1.0,2.25
1,EfficientNet-B0,8.5107,15.7117,0.9940,0.1457,3.0,2.0,1.0,3.0,2.25


In [11]:
# ----------------------------------------
# Section 10: Figure 9 - Accuracy-efficiency Pareto analysis
# ----------------------------------------

fig, ax = plt.subplots(figsize=(7.5, 5))

sizes = table9_df["Model Size (MB)"].astype(float).values * 25

ax.scatter(
    table9_df["Avg. Pipeline Time (ms)"].astype(float),
    table9_df["PlantVillage F1"].astype(float),
    s=sizes,
    alpha=0.65
)

for _, row in table9_df.iterrows():
    ax.text(
        float(row["Avg. Pipeline Time (ms)"]) + 0.5,
        float(row["PlantVillage F1"]) + 0.0008,
        row["Model"]
    )

ax.set_xlabel("Average deployment pipeline time (ms)")
ax.set_ylabel("PlantVillage F1-score")
ax.set_title("Figure 9. Accuracy-Efficiency Pareto Analysis of Candidate CNN Architectures")
ax.grid(alpha=0.25)

save_figure(fig, "Figure_9_Accuracy_Efficiency_Pareto")

Saved figure: /kaggle/working/thesis_outputs/rq5_deployment/figures/Figure_9_Accuracy_Efficiency_Pareto.pdf


In [12]:
# ----------------------------------------
# Section 11: Figure 10 - Measured deployment pipeline latency breakdown
# ----------------------------------------

models_order = table9_df["Model"].tolist()

preprocess = table9_df["Avg. Preprocess Time (ms)"].astype(float).values
inference = table9_df["Avg. Inference Time (ms)"].astype(float).values
decision = table9_df["Avg. Decision Time (ms)"].astype(float).values

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(models_order, preprocess, label="Preprocessing")
ax.bar(models_order, inference, bottom=preprocess, label="Inference")
ax.bar(models_order, decision, bottom=preprocess + inference, label="Decision logic")

totals = preprocess + inference + decision

for i, total in enumerate(totals):
    ax.text(
        i,
        total,
        f"{total:.2f} ms",
        ha="center",
        va="bottom",
        fontsize=9
    )

ax.set_ylabel("Latency (ms)")
ax.set_title("Figure 10. Measured Deployment Pipeline Latency Breakdown Across Models")
ax.legend()
ax.grid(axis="y", alpha=0.25)

save_figure(fig, "Figure_10_Measured_Deployment_Pipeline_Latency_Breakdown")

Saved figure: /kaggle/working/thesis_outputs/rq5_deployment/figures/Figure_10_Measured_Deployment_Pipeline_Latency_Breakdown.pdf


In [14]:
# ----------------------------------------
# Section 12: Save RQ5 metadata
# ----------------------------------------

meta_dir = ensure_dir(OUTPUT_ROOT / "metadata")

rq5_meta = {
    "seed": SEED,
    "models_evaluated": MODEL_NAMES,
    "benchmark_batch_size": 1,
    "warmup_runs": 10,
    "timed_runs": 50,
    "num_classes": NUM_CLASSES,
    "measured_pipeline_components": [
        "preprocessing",
        "inference",
        "uncertainty_aware_decision_logic"
    ],
    "excluded_components": [
        "network_upload_time",
        "frontend_rendering_time",
        "grad_cam_generation_time"
    ],
    "decision_rule_inputs": [
        "confidence",
        "normalized_entropy"
    ],
    "note": (
        "Latency was measured using batch size 1 to reflect single-image deployment. "
        "The measured pipeline includes preprocessing, neural network inference, and "
        "uncertainty-aware rule-based decision logic. Network upload, frontend rendering, "
        "and Grad-CAM generation latency were excluded because they were not benchmarked "
        "in this deployment notebook."
    )
}

meta_path = meta_dir / "rq5_metadata.json"

with open(meta_path, "w") as f:
    json.dump(rq5_meta, f, indent=2)

print("RQ5 metadata saved successfully")
print(f"Metadata path: {meta_path}")

RQ5 metadata saved successfully
Metadata path: /kaggle/working/thesis_outputs/rq5_deployment/metadata/rq5_metadata.json


In [15]:
# ----------------------------------------
# Section 13: Create ZIP archive
# ----------------------------------------

zip_path = OUTPUT_ROOT.parent / "06_rq5_deployment_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_ROOT.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(OUTPUT_ROOT))

print("ZIP archive created successfully")
print(f"ZIP file: {zip_path}")
print("06_rq5_deployment notebook completed successfully")

ZIP archive created successfully
ZIP file: /kaggle/working/thesis_outputs/06_rq5_deployment_outputs.zip
06_rq5_deployment notebook completed successfully
